# Personalized Real Estate Agent: "HomeMatch"

#### Overview
"HomeMatch" is a personalized real estate agent application designed to match potential buyers with properties based on their unique preferences. It leverages AI-powered language models and vector embeddings to create an engaging and customized home-buying experience.

#### Note on API Key Budget Constraints
The restrictive budget settings on the provided API key caused repeated "budget exceeded" errors, preventing me from using the real environment as intended. Despite multiple attempts, I had to implement mock solutions to simulate the functionality. I kindly request that this be considered for evaluation, as the limitations were due to the API configuration and beyond my control.

In [ ]:
# pip install pandas

In [3]:
import os
import pandas as pd
import numpy as np

from langchain.chat_models import ChatOpenAI
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from langchain.chains.question_answering import load_qa_chain
from langchain.document_loaders import TextLoader
from langchain.schema import Document
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores import Chroma
from langchain.text_splitter import CharacterTextSplitter

In [13]:
# os.environ["OPENAI_API_KEY"] = "voc-20308085881266773643716672d13f4d34476.56087839"
# os.environ["OPENAI_API_BASE"] = "https://openai.vocareum.com/v1"

# llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0.7, max_tokens=500)# 

In [17]:
# Generating Real Estate Listings

class MockLLM:
    def __call__(self, messages):
        # Simulate a response based on the user message content
        user_content = messages[1]["content"]
        if "Generate 10 real estate listings" in user_content:
            return """
            Neighborhood: Green Oaks
            Price: $800,000
            Bedrooms: 3
            Bathrooms: 2
            House Size: 2,000 sqft
            Description: Eco-friendly home with solar panels and hardwood floors.
            Neighborhood Description: A close-knit community with bike paths and organic markets.
            
            Neighborhood: Pine Meadows
            Price: $950,000
            Bedrooms: 4
            Bathrooms: 3
            House Size: 2,400 sqft
            Description: Spacious home with a modern kitchen and large backyard.
            Neighborhood Description: Family-friendly with excellent schools and nearby parks.
            
            Neighborhood: Riverfront Estates
            Price: $1,200,000
            Bedrooms: 5
            Bathrooms: 4
            House Size: 3,500 sqft
            Description: Luxurious waterfront home with a private dock and stunning views.
            Neighborhood Description: Exclusive community with access to boating and fishing.
            """
        else:
            return "Mock response: Task completed successfully."

# Initialize the mock LLM
llm = MockLLM()

# Define the prompt
listing_prompt = """
Generate 10 real estate listings in the following format:
Neighborhood: [name]
Price: [price]
Bedrooms: [number]
Bathrooms: [number]
House Size: [size in sqft]
Description: [description of the property]
Neighborhood Description: [description of the neighborhood]
"""

# Define messages
messages = [
    {"role": "system", "content": "You are a real estate assistant."},
    {"role": "user", "content": listing_prompt}
]

# Generate the mock response
response = llm(messages)

# Process the response
listings = response.split("\n\n")  # Split into individual listings
listings = [listing.strip() for listing in listings if listing.strip()]  # Clean up entries

# Print the mock listings
for i, listing in enumerate(listings, 1):
    print(f"Listing {i}:\n{listing}\n")

Listing 1:
Neighborhood: Green Oaks
            Price: $800,000
            Bedrooms: 3
            Bathrooms: 2
            House Size: 2,000 sqft
            Description: Eco-friendly home with solar panels and hardwood floors.
            Neighborhood Description: A close-knit community with bike paths and organic markets.
            
            Neighborhood: Pine Meadows
            Price: $950,000
            Bedrooms: 4
            Bathrooms: 3
            House Size: 2,400 sqft
            Description: Spacious home with a modern kitchen and large backyard.
            Neighborhood Description: Family-friendly with excellent schools and nearby parks.
            
            Neighborhood: Riverfront Estates
            Price: $1,200,000
            Bedrooms: 5
            Bathrooms: 4
            House Size: 3,500 sqft
            Description: Luxurious waterfront home with a private dock and stunning views.
            Neighborhood Description: Exclusive community with access

In [19]:
# Save listings to a file
with open("listings.txt", "w") as file:
    file.write("\n\n".join(listings))

print("Listings saved to listings.txt.")

Listings saved to listings.txt.


In [24]:
# Load listings from the text file
with open("listings.txt", "r") as file:
    listings = file.read().split("\n\n")

# Wrap listings into Document objects
documents = [Document(page_content=listing) for listing in listings]

# Split long listings into manageable chunks
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
split_listings = text_splitter.split_documents(documents)

# Print split listings to verify
for doc in split_listings:
    print(f"Chunk: {doc.page_content}\n")

Chunk: Neighborhood: Green Oaks
            Price: $800,000
            Bedrooms: 3
            Bathrooms: 2
            House Size: 2,000 sqft
            Description: Eco-friendly home with solar panels and hardwood floors.
            Neighborhood Description: A close-knit community with bike paths and organic markets.
            
            Neighborhood: Pine Meadows
            Price: $950,000
            Bedrooms: 4
            Bathrooms: 3
            House Size: 2,400 sqft
            Description: Spacious home with a modern kitchen and large backyard.
            Neighborhood Description: Family-friendly with excellent schools and nearby parks.
            
            Neighborhood: Riverfront Estates
            Price: $1,200,000
            Bedrooms: 5
            Bathrooms: 4
            House Size: 3,500 sqft
            Description: Luxurious waterfront home with a private dock and stunning views.
            Neighborhood Description: Exclusive community with access to 

In [34]:
# Mock embeddings and documents

class MockEmbeddings:
    def embed_documents(self, texts):
        # Generate random embeddings for each document
        return [np.random.rand(768).tolist() for _ in texts]

    def embed_query(self, query):
        # Generate a random embedding for the query
        return np.random.rand(768).tolist()
    
embeddings = MockEmbeddings()

listings = [
    """Neighborhood: Green Oaks
Price: $800,000
Bedrooms: 3
Bathrooms: 2
House Size: 2,000 sqft

Description: Welcome to this eco-friendly oasis nestled in the heart of Green Oaks. This charming 3-bedroom, 2-bathroom home boasts energy-efficient features such as solar panels and a well-insulated structure. Natural light floods the living spaces, highlighting the beautiful hardwood floors and eco-conscious finishes. The open-concept kitchen and dining area lead to a spacious backyard with a vegetable garden, perfect for the eco-conscious family. Embrace sustainable living without compromising on style in this Green Oaks gem.

Neighborhood Description: Green Oaks is a close-knit, environmentally-conscious community with access to organic grocery stores, community gardens, and bike paths. Take a stroll through the nearby Green Oaks Park or grab a cup of coffee at the cozy Green Bean Cafe. With easy access to public transportation and bike lanes, commuting is a breeze."""
]

documents = [Document(page_content=listing) for listing in listings]

# Create ChromaDB
db = Chroma.from_documents(documents, embeddings)

# Semantic Search Implementation

query = "I want a 3-bedroom house in a quiet neighborhood with good schools."
similar_listings = db.similarity_search(query, k=1)

# Print the matched listings
for i, result in enumerate(similar_listings, 1):
    print(f"Matched Listing {i}:\n{result.page_content}\n")

Matched Listing 1:
Neighborhood: Green Oaks
Price: $800,000
Bedrooms: 3
Bathrooms: 2
House Size: 2,000 sqft

Description: Welcome to this eco-friendly oasis nestled in the heart of Green Oaks. This charming 3-bedroom, 2-bathroom home boasts energy-efficient features such as solar panels and a well-insulated structure. Natural light floods the living spaces, highlighting the beautiful hardwood floors and eco-conscious finishes. The open-concept kitchen and dining area lead to a spacious backyard with a vegetable garden, perfect for the eco-conscious family. Embrace sustainable living without compromising on style in this Green Oaks gem.

Neighborhood Description: Green Oaks is a close-knit, environmentally-conscious community with access to organic grocery stores, community gardens, and bike paths. Take a stroll through the nearby Green Oaks Park or grab a cup of coffee at the cozy Green Bean Cafe. With easy access to public transportation and bike lanes, commuting is a breeze.



In [35]:
# Personalized Results

def mock_personalize_listing(listing, preferences):
    return f"{listing}\n\nPersonalized Note: Based on your preferences, this listing highlights its {preferences}."

# Example of personalization
preferences = "eco-friendly features and good school district"
for listing in similar_listings:
    print(mock_personalize_listing(listing.page_content, preferences))

Neighborhood: Green Oaks
Price: $800,000
Bedrooms: 3
Bathrooms: 2
House Size: 2,000 sqft

Description: Welcome to this eco-friendly oasis nestled in the heart of Green Oaks. This charming 3-bedroom, 2-bathroom home boasts energy-efficient features such as solar panels and a well-insulated structure. Natural light floods the living spaces, highlighting the beautiful hardwood floors and eco-conscious finishes. The open-concept kitchen and dining area lead to a spacious backyard with a vegetable garden, perfect for the eco-conscious family. Embrace sustainable living without compromising on style in this Green Oaks gem.

Neighborhood Description: Green Oaks is a close-knit, environmentally-conscious community with access to organic grocery stores, community gardens, and bike paths. Take a stroll through the nearby Green Oaks Park or grab a cup of coffee at the cozy Green Bean Cafe. With easy access to public transportation and bike lanes, commuting is a breeze.

Personalized Note: Based o